# Предсказание исхода матчей Dota 2

Цель проекта — предсказать вероятность победы команды **Radiant**.

В итоговой модели используются:
- дата и день недели;
- регион;
- средний рейтинг игроков;
- индикатор пропуска MMR;
- выбранные героями составы Radiant и Dire.

Метрика качества — коэффициент Gini:

`Gini = 2 * ROC-AUC - 1`.

## 1. Импорты

In [ ]:
from pathlib import Path
import sys

sys.path.append("..")

import optuna
import pandas as pd
from sklearn.linear_model import LogisticRegression

from src.features import (
    build_full_training_matrix,
    build_validation_matrices,
    gini_score,
    prepare_basic_features,
)

## 2. Загрузка данных

In [ ]:
data_dir = Path("../data")

df_train = pd.read_csv(data_dir / "matches_df_train.csv")
df_test = pd.read_csv(data_dir / "matches_df_test.csv")
player_df = pd.read_csv(data_dir / "player_df.csv")
heroes_df = pd.read_csv(data_dir / "Constants.Heroes.csv")

print("train:", df_train.shape)
print("test:", df_test.shape)
df_train.head()

## 3. Генерация признаков

Из даты выделяются `day` и `dayofweek`. Регион и день недели кодируются One-Hot Encoding.

Для `avg_mmr` используется медианное заполнение пропусков, а в модель передаются:
- `mmr_missing`;
- `sqrt_mmr`.

Выбранные герои кодируются разреженным вектором: `+1` для Radiant и `-1` для Dire.

In [ ]:
df_train, df_test = prepare_basic_features(
    df_train,
    df_test,
)

matrices = build_validation_matrices(
    df_train,
    df_test,
    player_df,
    heroes_df,
)

print("X_train:", matrices["X_train"].shape)
print("X_valid:", matrices["X_valid"].shape)
print("X_test:", matrices["X_test"].shape)

## 4. Базовая модель

In [ ]:
baseline = LogisticRegression(max_iter=3000)
baseline.fit(
    matrices["X_train"],
    matrices["y_train"],
)

valid_prediction = baseline.predict_proba(
    matrices["X_valid"]
)[:, 1]

print(
    "Validation Gini:",
    gini_score(
        matrices["y_valid"],
        valid_prediction,
    ),
)

## 5. Подбор гиперпараметров Optuna

In [ ]:
def objective(trial):
    params = {
        "C": trial.suggest_float(
            "C",
            2.0,
            4.0,
            log=True,
        ),
        "solver": "lbfgs",
        "max_iter": trial.suggest_int(
            "max_iter",
            2500,
            5000,
            step=500,
        ),
    }

    model = LogisticRegression(**params)
    model.fit(
        matrices["X_train"],
        matrices["y_train"],
    )

    prediction = model.predict_proba(
        matrices["X_valid"]
    )[:, 1]

    return gini_score(
        matrices["y_valid"],
        prediction,
    )


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=8)

print("Best params:", study.best_params)
print("Best validation Gini:", study.best_value)

## 6. Финальная модель

После выбора гиперпараметров модель переобучается на всей обучающей выборке и формирует вероятности для test.

In [ ]:
X_train_full, y_train_full = build_full_training_matrix(
    df_train,
    matrices["mmr_median"],
    matrices["heroes_encoder"],
)

final_model = LogisticRegression(**study.best_params)
final_model.fit(
    X_train_full,
    y_train_full,
)

predictions = final_model.predict_proba(
    matrices["X_test"]
)[:, 1]

submission = pd.DataFrame({
    "ID": matrices["test"]["match_id"],
    "Value": predictions,
})

submission.head()

In [ ]:
submission.to_csv(
    "../submission.csv",
    index=False,
)